# Notebook 4: the same pipeline for every RpSy well across the USA

**What this notebook does:** takes every well in the RpSy site list (Malakar et al. 2025), not just the Northeast, and runs the notebook 1 steps on all of them: Daymet precipitation, DTW pairing, and the cumulative precipitation vs cumulative recharge curves with their distance from the 1:1 line.
Run top to bottom. Only **cell 1** (settings) needs editing.

| Cell | What it does |
|---|---|
| 1 | Settings: file paths and the same DTW and curve settings as notebook 1 |
| 2 | Loads every well in the site list, finds its state, keeps wells with at least 5 years of RpSy, and flags the Northeast wells and the original 163. Saves `wells_usa.csv` |
| 3 | Downloads Daymet precipitation for wells that do not have it yet (wells from notebook 1 are reused) |
| 4 | The DTW functions (identical to notebook 1) |
| 5 | Runs DTW for every well and writes one event table each to `dtw_event_tables_rpsy/` (wells already done in notebook 1 are skipped) |
| 6 | Builds the cumulative curves (storms of at least 5 mm, smallest to largest) and saves the distance from the 1:1 line, seasonal shares and pairing statistics |
| 7 | Quick checks: wells per state, and median curves for all wells, the Northeast, and the rest of the USA |

**Outputs (all new files, `master_well_summary.csv` is never written):** `wells_usa.csv`, `cumulative_gaps_annual_usa.csv`, `cumulative_gaps_seasonal_usa.csv`, `seasonal_shares_usa.csv`, plus event tables in `dtw_event_tables_rpsy/`.

**Notes:** the first run downloads Daymet for every new well and runs DTW on each, so it can take a long time. It is safe to stop and re-run: finished wells are skipped.
Because the curves use the same settings as notebook 1, the Northeast wells here come out the same as in notebook 1.

In [ ]:
# ===== CELL 1: SETTINGS (the only cell you need to edit) =====
import os, io, time, glob
import numpy as np, pandas as pd, geopandas as gpd, requests
from dtw import dtw                                   # pip install dtw-python

# ---- input files ----
WELL_SITES   = r"C:\Users\romin\OneDrive\Groundwater\RpSy Data\Site information for all selected wells.xlsx"
STATES_PATH  = r"C:\Users\romin\OneDrive\Groundwater\cb_2022_us_state_500k"
RECHARGE_DIR = "C:/Users/romin/OneDrive/Groundwater/RpSy Data"      # {well_id}.csv with columns: Date, RpSy (m)   (Malakar et al. 2025)
DAYMET_DIR   = "./daymet"                                           # same folder notebook 1 uses, so wells it already downloaded are reused

# ---- output files (all new; master_well_summary.csv is never written) ----
EVENT_DIR         = "dtw_event_tables_rpsy"                         # same folder notebook 1 uses; wells already done there are skipped
WELLS_CSV         = "wells_usa.csv"                                 # every RpSy well that passes the record-length rule
GAPS_ANNUAL_CSV   = "cumulative_gaps_annual_usa.csv"
GAPS_SEASONAL_CSV = "cumulative_gaps_seasonal_usa.csv"
SHARES_CSV        = "seasonal_shares_usa.csv"

# ---- which wells ----
MIN_RECORD_YEARS = 5                                                # RpSy record must be at least this long (same rule as notebook 1)
NE_STATES = ["Connecticut", "Maine", "Massachusetts", "New Hampshire", "Rhode Island",
             "Vermont", "New Jersey", "New York", "Pennsylvania"]
ORIGINAL_FLAG_CSV     = "rpsy_all_summary.csv"                      # read only: its original_163 column marks the original 163 wells
ORIGINAL_IDS_FALLBACK = "master_well_summary.csv"                   # read only: used if the file above is missing
DAYMET_START_YEAR, DAYMET_END_YEAR = 1980, 2024

# ---- DTW pairing (same as notebook 1) ----
PRECIP_THRESHOLD = 2.5                                              # mm/day; a storm is consecutive days above this
MIN_LAG, MAX_LAG = 0, 5                                             # days recharge may lag the precipitation

# ---- cumulative curves (same as notebook 1) ----
MIN_STORM  = 5                                                      # mm per storm
MIN_EVENTS = 20                                                     # minimum storms for a well (or well-season) to get a curve
GRID_STEP  = 0.01
SEASONS = ["Winter", "Spring", "Summer", "Fall"]
MONTH2SEASON = {12: "Winter", 1: "Winter", 2: "Winter", 3: "Spring", 4: "Spring", 5: "Spring",
                6: "Summer", 7: "Summer", 8: "Summer", 9: "Fall", 10: "Fall", 11: "Fall"}
GRID = np.round(np.arange(GRID_STEP, 1 - GRID_STEP / 2, GRID_STEP), 4)
GCOLS = [f"gap_{int(round(g * 100)):02d}" for g in GRID]            # gap_01 ... gap_99

os.makedirs(EVENT_DIR, exist_ok=True); os.makedirs(DAYMET_DIR, exist_ok=True)

In [ ]:
# ===== CELL 2: ALL RpSy WELLS ACROSS THE USA =====
sites = pd.read_excel(WELL_SITES).rename(columns={"ID": "well_id", "Lat": "lat", "Long": "lon", "depth (m)": "well_depth_m"})
sites["well_id"] = sites["well_id"].astype(str)
sites = sites.drop_duplicates("well_id").reset_index(drop=True)

# state of each well (left join, so a well that falls just outside a polygon is kept with state "unknown")
states = gpd.read_file(STATES_PATH).to_crs(epsg=4326)
pts = gpd.GeoDataFrame(sites, geometry=gpd.points_from_xy(sites["lon"], sites["lat"]), crs="EPSG:4326")
pts = gpd.sjoin(pts, states[["NAME", "geometry"]], how="left", predicate="within").drop(columns=["index_right"])
pts = pts.drop_duplicates("well_id").rename(columns={"NAME": "state"})
pts["state"] = pts["state"].fillna("unknown")

# record dates from each well's RpSy file
start, end = [], []
for wid in pts["well_id"]:
    f = f"{RECHARGE_DIR}/{wid}.csv"
    if not os.path.exists(f):
        start.append(pd.NaT); end.append(pd.NaT); continue
    d = pd.to_datetime(pd.read_csv(f, usecols=["Date"])["Date"])
    start.append(d.min()); end.append(d.max())
pts["start_date"], pts["end_date"] = start, end
pts["record_length_yr"] = (pts["end_date"] - pts["start_date"]).dt.days / 365.25

n_sites = len(pts)
n_file = int(pts["start_date"].notna().sum())
wells = pts[pts["record_length_yr"] >= MIN_RECORD_YEARS].copy()
wells["northeast"] = wells["state"].isin(NE_STATES)

# mark the original 163 wells (read only)
orig = None
if os.path.exists(ORIGINAL_FLAG_CSV):
    t = pd.read_csv(ORIGINAL_FLAG_CSV, dtype={"well_id": str})
    orig = set(t.loc[t["original_163"].astype(str).str.lower().isin(["true", "1", "1.0"]), "well_id"])
elif os.path.exists(ORIGINAL_IDS_FALLBACK):
    orig = set(pd.read_csv(ORIGINAL_IDS_FALLBACK, dtype={"well_id": str})["well_id"])
wells["original_163"] = wells["well_id"].isin(orig) if orig is not None else False

wells = wells[["well_id", "lat", "lon", "state", "well_depth_m", "start_date", "end_date",
               "record_length_yr", "northeast", "original_163"]].reset_index(drop=True)
wells.to_csv(WELLS_CSV, index=False)

print(f"wells in the site list: {n_sites}")
print(f"with an RpSy file: {n_file}")
print(f"with at least {MIN_RECORD_YEARS} years of RpSy: {len(wells)}")
print(f"  in the Northeast states: {int(wells['northeast'].sum())}")
if orig is None:
    print("  original 163 not marked (no file found)")
else:
    print(f"  original 163 found: {int(wells['original_163'].sum())} of {len(orig)}")
    lost = sorted(orig - set(wells["well_id"]))
    if lost: print("  original wells missing from this list:", lost)
print(f"  state unknown: {int((wells['state'] == 'unknown').sum())}")
print(f"\nsaved {WELLS_CSV}\n")
print(wells["state"].value_counts().to_string())

In [ ]:
# ===== CELL 3: DAILY PRECIPITATION (Daymet, 1 km). Only downloads wells that are not already in DAYMET_DIR =====
# Wells already downloaded for notebook 1 are reused. Run again if it stops; it picks up where it left off.
def fetch_daymet(lat, lon, start_year=DAYMET_START_YEAR, end_year=DAYMET_END_YEAR, retries=3):
    url = "https://daymet.ornl.gov/single-pixel/api/data"
    params = dict(lat=round(lat, 5), lon=round(lon, 5), vars="prcp",
                  start=f"{start_year}-01-01", end=f"{end_year}-12-31", format="csv")
    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, timeout=120)
            r.raise_for_status()
            lines = r.text.splitlines()
            i0 = next(i for i, l in enumerate(lines) if l.startswith("year,yday"))
            return pd.read_csv(io.StringIO("\n".join(lines[i0:])))
        except (requests.RequestException, StopIteration) as e:
            if attempt == retries - 1:
                print(f"  Daymet failed ({lat}, {lon}): {e}")
                return None
            time.sleep(5 * (attempt + 1))

def ensure_daymet(wid, lat, lon):
    p = f"{DAYMET_DIR}/{wid}.csv"
    if os.path.exists(p):
        return True
    dm = fetch_daymet(lat, lon)
    if dm is None:
        return False
    dm.to_csv(p, index=False); time.sleep(1)
    return True

missing = [w for w in wells["well_id"] if not os.path.exists(f"{DAYMET_DIR}/{w}.csv")]
print(f"Daymet files already here: {len(wells) - len(missing)} | to download: {len(missing)}")
todo = wells[wells["well_id"].isin(missing)]
failed_dm = []
for i, (_, w) in enumerate(todo.iterrows()):
    if not ensure_daymet(w["well_id"], w["lat"], w["lon"]):
        failed_dm.append(w["well_id"])
    if (i + 1) % 25 == 0 or i + 1 == len(todo):
        print(f"\rDaymet {i + 1}/{len(todo)}", end="", flush=True)
print(f"\nDaymet failed for {len(failed_dm)} wells" + (f": {failed_dm}" if failed_dm else ""))

In [ ]:
# ===== CELL 4: DTW PAIRING FUNCTIONS =====
# Recharge (query) is warped onto precipitation (reference) with a causal window: recharge may sit MIN_LAG..MAX_LAG days AFTER the rain.
# Both series are divided by their own standard deviation first so neither dominates the alignment.
def get_events(values, threshold):
    """Start/end index pairs for runs of consecutive values above the threshold (= one storm)."""
    above = (values > threshold).astype(int)
    padded = np.concatenate(([0], above, [0]))
    diff = np.diff(padded)
    starts = np.where(diff == 1)[0]
    ends = np.where(diff == -1)[0] - 1
    return starts, ends

def precip_recharge_event_table(df, precip_col, recharge_col, alignment, precip_threshold=0.0):
    """One row per storm: its precipitation, plus the recharge that the DTW alignment matched to it (NaN = unpaired storm)."""
    dates = df.index
    precip_vals = df[precip_col].values
    recharge_vals = df[recharge_col].values
    starts, ends = get_events(precip_vals, precip_threshold)

    rows = []
    prev_recharge_idx = None
    for s, e in zip(starts, ends):
        precip_idx = np.arange(s, e + 1)
        mask = np.isin(alignment.index2, precip_idx)
        recharge_idx = np.unique(alignment.index1[mask])

        if prev_recharge_idx is not None and recharge_idx.size > 0:
            recharge_idx = recharge_idx[recharge_idx > prev_recharge_idx.max()]     # a recharge day is only given to one storm

        row = {"precip_start_date": dates[s], "precip_end_date": dates[e],
               "precip_total": precip_vals[precip_idx].sum(), "precip_peak": precip_vals[precip_idx].max()}
        if recharge_idx.size == 0:
            row.update({"recharge_start_date": pd.NaT, "recharge_end_date": pd.NaT,
                        "recharge_total": np.nan, "recharge_peak": np.nan, "n_recharge_days": 0})
        else:
            row.update({"recharge_start_date": dates[recharge_idx.min()], "recharge_end_date": dates[recharge_idx.max()],
                        "recharge_total": recharge_vals[recharge_idx].sum(), "recharge_peak": recharge_vals[recharge_idx].max(),
                        "n_recharge_days": recharge_idx.size})
        rows.append(row)
        prev_recharge_idx = recharge_idx if recharge_idx.size > 0 else prev_recharge_idx
    return pd.DataFrame(rows)

def run_dtw_pipeline(site_id, recharge_dir=RECHARGE_DIR, daymet_dir=DAYMET_DIR, out_dir=EVENT_DIR,
                     precip_threshold=PRECIP_THRESHOLD, min_lag=MIN_LAG, max_lag=MAX_LAG):
    recharge_path, precip_path = f"{recharge_dir}/{site_id}.csv", f"{daymet_dir}/{site_id}.csv"
    if not os.path.exists(recharge_path) or not os.path.exists(precip_path):
        return None, "missing file"

    df_recharge, df_precip = pd.read_csv(recharge_path), pd.read_csv(precip_path)
    df_precip["Date"] = pd.to_datetime(df_precip["year"] * 1000 + df_precip["yday"], format="%Y%j")
    df_recharge["Date"] = pd.to_datetime(df_recharge["Date"])
    df_recharge.set_index("Date", inplace=True)
    df_precip.set_index("Date", inplace=True)

    df = pd.merge(df_recharge, df_precip, on="Date", how="inner")
    df.drop(columns=["year", "yday"], inplace=True, errors="ignore")
    if len(df) < 30:
        return None, "too short"

    recharge, precip = df["RpSy (m)"].values, df["prcp (mm/day)"].values
    if np.std(recharge) == 0 or np.std(precip) == 0:
        return None, "zero variance"
    recharge_norm, precip_norm = recharge / np.std(recharge), precip / np.std(precip)

    def causal_window(iw, jw, query_size, reference_size, min_lag=min_lag, max_lag=max_lag, **kwargs):
        lag = iw - jw
        ok = lag >= min_lag
        if max_lag is not None:
            ok = ok & (lag <= max_lag)
        return ok

    try:
        alignment = dtw(recharge_norm, precip_norm, step_pattern="symmetric2", window_type=causal_window,
                        window_args={"min_lag": min_lag, "max_lag": max_lag}, keep_internals=True,
                        open_begin=False, open_end=False)
    except Exception as e:
        return None, f"dtw failed: {e}"

    event_table = precip_recharge_event_table(df, "prcp (mm/day)", "RpSy (m)", alignment, precip_threshold=precip_threshold)
    event_table.to_csv(f"{out_dir}/{site_id}_dtw_event_table.csv", index=False)
    return event_table, "ok"

In [ ]:
# ===== CELL 5: RUN DTW FOR EVERY WELL (resumable: wells that already have an event table are skipped) =====
done, failed = 0, {}
t0 = time.time()
for i, wid in enumerate(wells["well_id"]):
    if os.path.exists(f"{EVENT_DIR}/{wid}_dtw_event_table.csv"):
        done += 1
    else:
        table, status = run_dtw_pipeline(wid)
        if table is not None: done += 1
        else: failed[wid] = status
    if (i + 1) % 25 == 0 or i + 1 == len(wells):
        print(f"\rwell {i + 1}/{len(wells)} | {(time.time() - t0) / 60:.1f} min", end="", flush=True)
print(f"\n\nEvent tables ready: {done} of {len(wells)} | failed: {len(failed)}")
if failed:
    from collections import Counter
    print("Reasons:", dict(Counter(failed.values())))
    print(failed)

In [ ]:
# ===== CELL 6: CUMULATIVE PRECIPITATION vs CUMULATIVE RECHARGE, DISTANCE FROM THE 1:1 LINE =====
# Storms >= MIN_STORM are ranked smallest -> largest. x = cumulative fraction of precipitation, y = cumulative fraction of recharge.
# A storm with no DTW-matched recharge counts as 0 recharge. Distance from the 1:1 line = y - x (negative = large storms dominate).
def read_events(wid):
    cols = ["precip_start_date", "recharge_start_date"]
    ev = pd.read_csv(f"{EVENT_DIR}/{wid}_dtw_event_table.csv", parse_dates=cols)
    ev = ev[ev["precip_total"] >= MIN_STORM].copy()
    ev["paired"] = ev["recharge_total"].notna()
    ev["recharge_total"] = ev["recharge_total"].fillna(0)
    ev["season"] = ev["precip_start_date"].dt.month.map(MONTH2SEASON)
    return ev

def curve(ev):
    """(x, y) of the cumulative curve, starting at (0, 0)."""
    e = ev.sort_values("precip_total")
    x = np.concatenate(([0], (e["precip_total"].cumsum() / e["precip_total"].sum()).values))
    y = np.concatenate(([0], (e["recharge_total"].cumsum() / e["recharge_total"].sum()).values))
    return x, y

def gaps(ev):
    """Distance from the 1:1 line at every GRID point, or None if the well/season is too thin."""
    if len(ev) < MIN_EVENTS or ev["recharge_total"].sum() == 0:
        return None
    x, y = curve(ev)
    return np.interp(GRID, x, y) - GRID

def pairing_stats(ev):
    """How well DTW paired this well's storms (used later to check the depth-to-water result)."""
    e = ev.sort_values("precip_total")
    third = len(e) // 3
    paired = e[e["paired"]]
    lag = (paired["recharge_start_date"] - paired["precip_start_date"]).dt.days
    return dict(n_events=len(e), frac_unpaired=1 - e["paired"].mean(),
                unpaired_small=1 - e.iloc[:third]["paired"].mean(), unpaired_large=1 - e.iloc[-third:]["paired"].mean(),
                med_storm_paired=paired["precip_total"].median(), med_storm_unpaired=e.loc[~e["paired"], "precip_total"].median(),
                med_lag_days=lag.median(), med_recharge_days=paired["n_recharge_days"].median())

annual_rows, seasonal_rows, share_rows = [], [], []
for wid in wells["well_id"]:
    f = f"{EVENT_DIR}/{wid}_dtw_event_table.csv"
    if not os.path.exists(f): continue
    ev = read_events(wid)
    if len(ev) < MIN_EVENTS: continue

    g = gaps(ev)                                                              # annual curve (all seasons together)
    annual_rows.append({"well_id": wid, **pairing_stats(ev), **dict(zip(GCOLS, g if g is not None else [np.nan] * len(GRID)))})

    for s in SEASONS:                                                          # one curve per season
        es = ev[ev["season"] == s]
        g = gaps(es)
        if g is not None:
            seasonal_rows.append({"well_id": wid, "season": s, "n_events": len(es), **dict(zip(GCOLS, g))})

    tot_p, tot_r = ev["precip_total"].sum(), ev["recharge_total"].sum()      # each season's share of the well's total
    if tot_r > 0:
        for s in SEASONS:
            es = ev[ev["season"] == s]
            share_rows.append({"well_id": wid, "season": s, "recharge_share": es["recharge_total"].sum() / tot_r,
                               "precip_share": es["precip_total"].sum() / tot_p})

annual, seasonal, shares = pd.DataFrame(annual_rows), pd.DataFrame(seasonal_rows), pd.DataFrame(share_rows)
attrs = wells[["well_id", "state", "lat", "lon", "northeast", "original_163"]]
annual = attrs.merge(annual, on="well_id", how="inner")                       # carry state / location / flags with the annual table
annual.to_csv(GAPS_ANNUAL_CSV, index=False); seasonal.to_csv(GAPS_SEASONAL_CSV, index=False); shares.to_csv(SHARES_CSV, index=False)   # new files
print(f"annual curves: {annual['gap_50'].notna().sum()} wells | seasonal curves: {len(seasonal)} well-seasons "
      f"({seasonal['well_id'].nunique()} wells) | shares: {shares['well_id'].nunique()} wells")
print(f"saved: {GAPS_ANNUAL_CSV}, {GAPS_SEASONAL_CSV}, {SHARES_CSV}")

In [ ]:
# ===== CELL 7: QUICK CHECKS =====
print(f"wells with an annual curve: {annual['gap_50'].notna().sum()} of {len(wells)}")
print(annual.groupby("state").size().sort_values(ascending=False).head(15).to_string())

print("\nMedian distance from the 1:1 line at 25/50/75%, by season (negative = large storms dominate)")
sm = seasonal.merge(wells[["well_id", "northeast", "original_163"]], on="well_id")
for label, d in [("all wells", sm), ("northeast", sm[sm["northeast"]]), ("rest of USA", sm[~sm["northeast"]])]:
    print(f"\n{label} ({d['well_id'].nunique()} wells)")
    print(d.groupby("season")[["gap_25", "gap_50", "gap_75"]].median().reindex(SEASONS).round(3).to_string())

print("\nMedian share of the well's annual total (%), all wells")
print((shares.groupby("season")[["recharge_share", "precip_share"]].median().reindex(SEASONS) * 100).round(1).to_string())

if annual["original_163"].any():
    o = annual[annual["original_163"]]
    print(f"\noriginal 163 wells in this run: {len(o)} | median distance at 50% (annual): {o['gap_50'].median():.3f}")